# Synthetic Data Pipeline

This notebook runs the synthetic data generation pipeline.

### Before getting started:
- Ensure you have read `docs/*` and `README.md`
- Check that `params.py` and `config.py` are correct.
- Check the `call_LLM` and `red_write_data` functions in `processing.py` are correctly configured for your platform .
- Check that your input data exists and is correctly formatted.

First, import the required classes. 

In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [ ]:
from src.data_generator import generate_patients, generate_admissions, generate_journeys, generate_clinical_notes, add_augmentations, save_final_outputs
from datetime import datetime
from config.params import PARAMS

In [ ]:
# INSERT RUN NAME BELOW
# This will be saved in the journey dataset at the end of the notebook for evaluation purposes
run_name = "wp_20_generations"
current_time = datetime.now()
version_tag = current_time.strftime("%Y-%m-%d") + f"/{run_name}"
print(f"Run name: {run_name}\nDate: {current_time}\nVersion tag: {version_tag}")

## 1 - Get Patients Information and Admissions

- Generates a list of patients with corresponding admission reasons.


In [ ]:
patient_generator = generate_patients()
patients = await patient_generator.run(return_output = True)
patient_generator.write_patients_to_dataset()

In [ ]:
admission_generator = generate_admissions()
admissions = await admission_generator.run(return_output = True)
admission_generator.write_admissions_to_dataset()

## 2 - Generating and Filtering Journeys

- Generates, validates, and adds details to patient journeys.
- Filtering removes any document types that are not listed as possible event types in `params.py`.

In [ ]:
journey_generator = generate_journeys()
journeys = await journey_generator.run(return_outputs = True)
journey_generator.write_journeys_to_dataset()

## 3 - Generate and Validate Clinical Notes

- Uses LLMs to generate clinical notes. 
- Validates each note using an LLM Judge. 

In [ ]:
clinical_note_generator = generate_clinical_notes()
notes = await clinical_note_generator.run(return_output = True)
clinical_note_generator.write_patient_documents_to_dataset()

In [ ]:
import json
print(json.loads(notes[0][7]))

## 4 - Add Augmentations to clinical Notes

This section allows for the augmentatio of clinical notes by:

- Replacing long phrases with abbreviations.
- Adding typos.
- Adding signatures.

In [ ]:
augmentator = add_augmentations()
augmented_notes = await augmentator.run(True)
augmentator.write_final_documents_to_dataset()

## 5 - Write Clinical Notes to Dataset

- Writes the clinical notes to a dataset, alongside the patient journey and admission details.

In [ ]:
output_saver = save_final_outputs()
output_saver.run(run_name, current_time, version_tag)